# Vietnamese Corpus N-Gram Frequency Pipeline

This notebook prepares corpus-derived Vietnamese n-gram frequency lists for a visual lexical decision task. The goal is to support stimulus construction by estimating corpus frequency for lexical-item candidates with 1 to 4 syllables.

The notebook does not select final stimuli and does not generate pseudowords. It only cleans corpus text, tokenizes it at the syllable level, generates n-grams, applies the existing rule-based filtering logic, computes frequency and log frequency, and exports research-ready CSV files.


## Imports

These libraries support file traversal, text cleaning, n-gram counting, numerical transformation, and tabular export.


In [1]:
from pathlib import Path
import math
import re
from collections import Counter

import pandas as pd


## Paths

The pipeline reads raw `.txt` files from `data_sample` and writes processed frequency files to `data_sample/_processed`. Existing `_processed` folders are skipped during corpus traversal so exported CSV files are not accidentally read back into the corpus.


In [2]:
PROJECT_DIR = Path(r"D:\Research 2")
ROOT_PATH = PROJECT_DIR / "data_sample"
OUTPUT_DIR = ROOT_PATH / "_processed"

# Use None to process all available files in each source folder.
# Set an integer for quick testing if needed, but keep None for final exports.
MAX_FILES_PER_FOLDER = None

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Corpus root: {ROOT_PATH}")
print(f"Output directory: {OUTPUT_DIR}")


Corpus root: D:\Research 2\data_sample
Output directory: D:\Research 2\data_sample\_processed


## Cleaning Rules

The cleaning step removes obvious web noise, metadata, numbers, punctuation, and malformed strings while preserving Vietnamese letters and whitespace. Preserving whitespace is important because Vietnamese orthography separates syllables with spaces, and syllable count is the operational measure of word length in this project.


In [2]:
# Lines with these starts are usually recipe labels, metadata, or non-target text.
bad_line_starts = {
    "nguyên liệu",
    "thực hiện",
    "giá tham khảo",
}

# Strings that usually indicate metadata, email/web noise, or source boilerplate.
bad_line_contains = {
    "vnexpress",
    "subject",
    "sent",
    "from:",
    "to:",
    "http",
    "https",
    "www.",
    ".com",
    ".vn",
    "@",
}

# Stopwords are removed from unigram counts and used to reject n-grams
# that begin/end with function words or consist only of function words.
stopwords = {
    "và", "của", "là", "có", "cho", "các", "trong", "với",
    "được", "những", "một", "đã", "sẽ", "đến", "không",
    "như", "cũng", "này", "đó", "theo", "rằng", "tôi",
    "anh", "ta", "ấy", "chúng", "ở", "khi", "để", "ra", "từ", "vào",
}

bad_tokens = {
    "vnexpress", "net", "sent", "subject", "to", "from",
    "am", "pm", "com", "www", "http", "https", "gd", "dt",
}

vietnamese_letter_pattern = (
    r"[^\w\sàáạảãâầấậẩẫăằắặẳẵ"
    r"èéẹẻẽêềếệểễìíịỉĩ"
    r"òóọỏõôồốộổỗơờớợởỡ"
    r"ùúụủũưừứựửữỳýỵỷỹđ]"
)


def clean_line_for_project(line):
    """
    Clean one corpus line for the lexical decision project:
    - preserve Vietnamese letters with diacritics
    - remove numbers, punctuation, and web artifacts
    - preserve spaces because they mark syllable boundaries in Vietnamese orthography
    """
    line = line.strip().lower()

    if not line:
        return ""

    if line in bad_line_starts or any(line.startswith(x + ":") for x in bad_line_starts):
        return ""

    line = re.sub(r"\d+", " ", line)
    line = line.replace("-", " ")
    line = re.sub(vietnamese_letter_pattern, " ", line)
    line = line.replace("_", " ")
    line = re.sub(r"\s+", " ", line).strip()

    if any(x in line for x in ["img", "http", "www", "src", "width", "height", "border"]):
        return ""

    if any(x in line for x in bad_line_contains):
        return ""

    return line


## Tokenization

Vietnamese text is tokenized by whitespace after cleaning. In this project, these whitespace-separated units are treated as syllables for n-gram generation and syllable-length measurement.


In [3]:
def read_text_file(file_path):
    """Read a text file with fallback encodings to avoid stopping on mixed corpus encodings."""
    try:
        return file_path.read_text(encoding="utf-8").splitlines()
    except UnicodeDecodeError:
        try:
            return file_path.read_text(encoding="utf-16").splitlines()
        except UnicodeDecodeError:
            return file_path.read_text(encoding="utf-8", errors="ignore").splitlines()


def tokenize_lines(lines):
    cleaned_lines = []

    for line in lines:
        clean_line = clean_line_for_project(line)
        if clean_line:
            cleaned_lines.append(clean_line)

    text = " ".join(cleaned_lines)
    tokens = text.split()
    return [token for token in tokens if token not in bad_tokens]


## N-Gram Generation

N-grams are generated from consecutive syllable tokens. Unigrams correspond to 1-syllable candidates, bigrams to 2-syllable candidates, trigrams to 3-syllable candidates, and four-grams to 4-syllable candidates.


In [4]:
def generate_ngrams(tokens, n):
    """Generate consecutive n-grams from syllable tokens."""
    return [" ".join(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def is_valid_ngram(ngram):
    """Apply the existing stopword boundary logic to candidate n-grams."""
    words = ngram.split()

    if not words:
        return False

    if all(word in stopwords for word in words):
        return False

    if words[0] in stopwords or words[-1] in stopwords:
        return False

    return True


## Unigrams

Unigrams provide 1-syllable lexical-item candidates. Stopwords are removed because function words are not suitable as real-word stimuli in this lexical decision design.


In [5]:
def collect_unigrams(tokens):
    return [token for token in tokens if token not in stopwords]


## Bigrams

Bigrams are important because many Vietnamese lexical items contain two syllables. The existing filtering logic removes stopword-boundary n-grams and a small set of discourse-like or verb-like candidates.


In [6]:
def is_good_bigram(item):
    words = item.split()

    bad_phrases = {"tuy nhiên", "sau khi", "có thể"}
    if item in bad_phrases:
        return False

    verb_like = {"làm", "sử", "dụng", "thực", "hiện"}
    if any(word in verb_like for word in words):
        return False

    return True


def collect_bigrams(tokens):
    bigrams = generate_ngrams(tokens, 2)
    return [bigram for bigram in bigrams if is_valid_ngram(bigram) and is_good_bigram(bigram)]


## Trigrams

Trigrams support 3-syllable lexical-item candidates. The filtering follows the existing blacklist and stopword-boundary logic to reduce fragmented or function-like expressions.


In [7]:
bad_trigrams = {
    "bạn có thể",
    "giám đốc công",
    "đốc công ty",
    "báo người lao",
    "tiểu đường loại",
    "cơ quan điều",
    "quan điều tra",
    "tphcm trả lời",
    "lần đầu tiên",
    "trên địa bàn",
    "chứ không phải",
    "tại hà nội",
    "cơ quan chức",
    "quan chức năng",
    "quan quản lý",
    "cơ quan quản",
}


def is_good_trigram(item):
    words = item.split()

    if item in bad_trigrams:
        return False

    if words[0] in stopwords or words[-1] in stopwords:
        return False

    if item in {"bạn có thể", "trên thị trường", "trên thế giới", "chứ không phải"}:
        return False

    return True


def collect_trigrams(tokens):
    trigrams = generate_ngrams(tokens, 3)
    return [trigram for trigram in trigrams if is_valid_ngram(trigram) and is_good_trigram(trigram)]


## Four-Grams (New)

Four-grams provide 4-syllable candidates, which are needed because the experimental design treats syllable length as an independent variable. The filtering intentionally mirrors the trigram logic rather than introducing a drastically different rule set.


In [8]:
bad_fourgrams = {
    "chứ không phải là",
    "trên địa bàn thành",
    "cơ quan chức năng",
    "cơ quan quản lý",
    "bạn có thể sử",
    "có thể sử dụng",
}


def is_good_fourgram(item):
    words = item.split()

    if item in bad_fourgrams:
        return False

    if words[0] in stopwords or words[-1] in stopwords:
        return False

    return True


def collect_fourgrams(tokens):
    fourgrams = generate_ngrams(tokens, 4)
    return [fourgram for fourgram in fourgrams if is_valid_ngram(fourgram) and is_good_fourgram(fourgram)]


## Frequency Computation

This cell traverses the corpus, applies the cleaning and tokenization rules, collects 1- to 4-syllable candidates, and counts their corpus frequencies. These frequencies are the basis for the log frequency predictor used later in the lexical decision analysis.


In [10]:
all_unigrams = []
all_bigrams = []
all_trigrams = []
all_fourgrams = []
file_count = 0

for folder in ROOT_PATH.rglob("*"):
    if not folder.is_dir() or folder.name == "_processed":
        continue

    txt_files = sorted(folder.glob("*.txt"))
    if MAX_FILES_PER_FOLDER is not None:
        txt_files = txt_files[:MAX_FILES_PER_FOLDER]

    for file_path in txt_files:
        lines = read_text_file(file_path)
        tokens = tokenize_lines(lines)

        if not tokens:
            continue

        file_count += 1
        all_unigrams.extend(collect_unigrams(tokens))
        all_bigrams.extend(collect_bigrams(tokens))
        all_trigrams.extend(collect_trigrams(tokens))
        all_fourgrams.extend(collect_fourgrams(tokens))

freq_uni = Counter(all_unigrams)
freq_bi = Counter(all_bigrams)
freq_tri = Counter(all_trigrams)
freq_four = Counter(all_fourgrams)

print(f"Files read: {file_count}")
print(f"Unigram tokens collected: {len(all_unigrams)}")
print(f"Bigrams collected: {len(all_bigrams)}")
print(f"Trigrams collected: {len(all_trigrams)}")
print(f"Four-grams collected: {len(all_fourgrams)}")


Files read: 1186
Unigram tokens collected: 409401
Bigrams collected: 308246
Trigrams collected: 310230
Four-grams collected: 315124


## Filtering

The rule-based filters have already been applied during n-gram collection. This step converts frequency counters into data frames, assigns syllable length, computes log frequency, and standardizes the column format for later stimulus review.


In [11]:
def frequency_dataframe(counter, syllable_length):
    df = pd.DataFrame(counter.items(), columns=["lexical_item", "frequency"])
    df["syllable_length"] = syllable_length
    df["log_frequency"] = df["frequency"].apply(math.log)
    return df.sort_values("frequency", ascending=False).reset_index(drop=True)


df_uni = frequency_dataframe(freq_uni, 1)
df_bi = frequency_dataframe(freq_bi, 2)
df_tri = frequency_dataframe(freq_tri, 3)
df_four = frequency_dataframe(freq_four, 4)

df_all = pd.concat([df_uni, df_bi, df_tri, df_four], ignore_index=True)
df_all = df_all.sort_values(
    by=["syllable_length", "frequency"],
    ascending=[True, False],
).reset_index(drop=True)

expected_columns = ["lexical_item", "frequency", "syllable_length", "log_frequency"]
df_uni = df_uni[expected_columns]
df_bi = df_bi[expected_columns]
df_tri = df_tri[expected_columns]
df_four = df_four[expected_columns]
df_all = df_all[expected_columns]

print(df_all.head())


  lexical_item  frequency  syllable_length  log_frequency
0        người       6474                1       8.775549
1          bạn       5823                1       8.669571
2          chị       4021                1       8.299286
3         mình       3768                1       8.234300
4          yêu       3162                1       8.058960


## Export

The processed frequency tables are exported as UTF-8 CSV files with BOM for safer display in spreadsheet software on Windows. The combined file now includes 1- to 4-syllable candidates.


In [12]:
exports = {
    "unigrams_frequency.csv": df_uni,
    "bigrams_frequency.csv": df_bi,
    "trigrams_frequency.csv": df_tri,
    "fourgrams_frequency.csv": df_four,
    "all_1to4gram_frequency.csv": df_all,
}

for filename, dataframe in exports.items():
    path = OUTPUT_DIR / filename
    dataframe.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Saved {filename}: {len(dataframe)} rows")

print(f"\nFiles saved to: {OUTPUT_DIR}")


Saved unigrams_frequency.csv: 5995 rows
Saved bigrams_frequency.csv: 113088 rows


Saved trigrams_frequency.csv: 233180 rows


Saved fourgrams_frequency.csv: 277508 rows


Saved all_1to4gram_frequency.csv: 629771 rows

Files saved to: D:\Research 2\data_sample\_processed


## Summary

These summaries support research documentation and stimulus-construction planning. They show how many candidate items exist at each syllable length, which items are most frequent, and the observed frequency ranges before manual stimulus validation.


In [13]:
item_counts = (
    df_all.groupby("syllable_length")
    .size()
    .rename("item_count")
    .reset_index()
)

frequency_ranges = (
    df_all.groupby("syllable_length")
    .agg(
        min_frequency=("frequency", "min"),
        max_frequency=("frequency", "max"),
        min_log_frequency=("log_frequency", "min"),
        max_log_frequency=("log_frequency", "max"),
    )
    .reset_index()
)

print("Number of candidate items per syllable length:")
print(item_counts.to_string(index=False))

print("\nFrequency ranges per syllable length:")
print(frequency_ranges.to_string(index=False))

for syllable_length, dataframe in [
    (1, df_uni),
    (2, df_bi),
    (3, df_tri),
    (4, df_four),
]:
    print(f"\nTop 20 items with syllable_length = {syllable_length}:")
    print(dataframe.head(20).to_string(index=False))


Number of candidate items per syllable length:
 syllable_length  item_count
               1        5995
               2      113088
               3      233180
               4      277508

Frequency ranges per syllable length:
 syllable_length  min_frequency  max_frequency  min_log_frequency  max_log_frequency
               1              1           6474                0.0           8.775549
               2              1           1352                0.0           7.209340
               3              1            554                0.0           6.317165
               4              1             54                0.0           3.988984

Top 20 items with syllable_length = 1:
lexical_item  frequency  syllable_length  log_frequency
       người       6474                1       8.775549
         bạn       5823                1       8.669571
         chị       4021                1       8.299286
        mình       3768                1       8.234300
         yêu       3162 